In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


# Data Quality Checks
## Purpose
Validates bronze_orders after every pipeline ingest.
Raises a ValueError and stops execution if any check fails.
## Checks
1. Row count threshold: must have > 100 rows
2. Null audit: order_id and sales null % must be < 1%
3. Schema check: required columns must all be present
4. Referential integrity: all order_ids in fact_orders must exist in bronze_orders

In [9]:
# Read the Bronze table
df = spark.table("bronze_orders")

# Count the rows
row_count = df.count()
print(f"Row count: {row_count}")

# Set the minimum expected row count
ROW_THRESHOLD = 100
# Data quality check
if row_count < ROW_THRESHOLD:
    raise ValueError(
        f"QUALITY FAIL: Row count {row_count} is below the threshold of {ROW_THRESHOLD}."
    )
else:
    print(f"Row count check PASSED: {row_count} rows.")

StatementMeta(, dae75dce-a901-4a98-93f5-e460e59ed5c9, 11, Finished, Available, Finished, False)

Row count: 10194
Row count check PASSED: 10194 rows.


In [10]:
from pyspark.sql.functions import col

# Maximum allowed null percentage (1%)
NULL_THRESHOLD_PCT = 0.01

# Define quality checks
checks = {
    "order_id": col("order_id").isNull() | (col("order_id") == ""),
    "sales": col("sales").isNull()
}

failed = []

# Check each column
for col_name, condition in checks.items():
    null_count = df.filter(condition).count()
    null_pct = null_count / row_count

    status = "PASS" if null_pct < NULL_THRESHOLD_PCT else "FAIL"

    print(f"{col_name}: {null_count} nulls ({null_pct:.2%}) -- {status}")

    if status == "FAIL":
        failed.append(col_name)

# Final result
if failed:
    raise ValueError(
        f"QUALITY FAIL: Null threshold breached for columns: {', '.join(failed)}"
    )
else:
    print("Null check PASSED for all critical columns.")

StatementMeta(, dae75dce-a901-4a98-93f5-e460e59ed5c9, 12, Finished, Available, Finished, False)

order_id: 0 nulls (0.00%) -- PASS
sales: 0 nulls (0.00%) -- PASS
Null check PASSED for all critical columns.


In [11]:
print(df.columns)

StatementMeta(, dae75dce-a901-4a98-93f5-e460e59ed5c9, 13, Finished, Available, Finished, False)

['Row_ID', 'Order_ID', 'Order_Date', 'Ship_Date', 'Ship_Mode', 'Customer_ID', 'Customer_Name', 'Segment', 'Country/Region', 'City', 'State/Province', 'Postal_Code', 'Region', 'Product_ID', 'Category', 'Sub-Category', 'Product_Name', 'Sales', 'Quantity', 'Discount', 'Profit']


In [13]:
df.printSchema()

StatementMeta(, dae75dce-a901-4a98-93f5-e460e59ed5c9, 15, Finished, Available, Finished, False)

root
 |-- Row_ID: integer (nullable = true)
 |-- Order_ID: string (nullable = true)
 |-- Order_Date: string (nullable = true)
 |-- Ship_Date: string (nullable = true)
 |-- Ship_Mode: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country/Region: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State/Province: string (nullable = true)
 |-- Postal_Code: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product_ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Profit: double (nullable = true)



In [14]:
from pyspark.sql.functions import col

df = df.withColumnRenamed("Sub-Category", "sub_category")

StatementMeta(, dae75dce-a901-4a98-93f5-e460e59ed5c9, 16, Finished, Available, Finished, False)

In [10]:
print("Columns in DataFrame:")
for c in df.columns:
    print(c)

StatementMeta(, 73afc055-6e6d-4a76-a45a-9928d7b1a967, 12, Finished, Available, Finished, False)

Columns in DataFrame:
Row_ID
Order_ID
Order_Date
Ship_Date
Ship_Mode
Customer_ID
Customer_Name
Segment
Country/Region
City
State/Province
Postal_Code
Region
Product_ID
Category
sub_category
Product_Name
Sales
Quantity
Discount
Profit


In [17]:
REQUIRED_COLUMNS = [
    "order_id",
    "order_date",
    "ship_date",
    "ship_mode",
    "customer_id",
    "customer_name",
    "segment",
    "region",
    "product_id",
    "category",
    "sub_category",
    "sales",
    "quantity",
    "discount",
    "profit"
]

# Convert actual column names to lowercase
actual_columns = [c.lower() for c in df.columns]

# Check for missing columns
missing = [c for c in REQUIRED_COLUMNS if c not in actual_columns]

if missing:
    print("Available columns:", actual_columns)
    raise ValueError(f"QUALITY FAIL: missing required columns: {missing}")

print(f"Schema check PASSED: all {len(REQUIRED_COLUMNS)} required columns present.")

StatementMeta(, dae75dce-a901-4a98-93f5-e460e59ed5c9, 19, Finished, Available, Finished, False)

Schema check PASSED: all 15 required columns present.


In [18]:
from pyspark.sql.functions import broadcast

try:
    # Read fact_orders table
    df_fact = spark.table("fact_orders")

    # Get distinct order IDs from bronze_orders
    bronze_ids = df.select("order_id").distinct()

    # Find orphan records in fact_orders
    orphan_fact_rows = (
        df_fact.join(
            broadcast(bronze_ids),
            on="order_id",
            how="left_anti"
        ).count()
    )

    # Check referential integrity
    if orphan_fact_rows > 0:
        raise ValueError(
            f"QUALITY FAIL: {orphan_fact_rows} rows in fact_orders have no matching order_id in bronze_orders."
        )
    else:
        print(
            "Referential integrity check PASSED: "
            "All fact_orders rows have a matching bronze_orders order_id."
        )

except Exception as e:
    if "Table or view not found" in str(e):
        print("fact_orders not yet created -- skipping referential integrity check.")
    else:
        raise



StatementMeta(, dae75dce-a901-4a98-93f5-e460e59ed5c9, 20, Finished, Available, Finished, False)

Referential integrity check PASSED: All fact_orders rows have a matching bronze_orders order_id.
